# Data & training pipeline walkthrough

Walks through `src/data.py`'s weak-supervision labeling
(`assign_aspect`, `stars_to_sentiment`) and runs a short, CPU-feasible
training loop via `src/train.py`'s `Trainer` to demonstrate the training
path without requiring a full `python main.py train` run.

The dataset cells need network access to Hugging Face
(`McAuley-Lab/Amazon-Reviews-2023`) — if that's unavailable they print a
message and the notebook still runs to completion.

In [1]:
import os
from pathlib import Path

# Notebooks live in notebooks/; every relative path in this repo
# (config.yaml, V2/config.yaml, models/, ...) assumes the process cwd
# is the repo root, same as running `python main.py ...` from the shell.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("cwd:", Path.cwd())

cwd: C:\GitHub\NLPTransformerAnalysis


In [2]:
from src.data import load_config, ASPECT_KEYWORDS, assign_aspect, stars_to_sentiment

config = load_config("config.yaml")
print("Aspects:   ", config["aspects"])
print("Sentiments:", config["sentiments"])

Aspects:    {0: 'quality', 1: 'usability', 2: 'value', 3: 'shipping', 4: 'customer_service'}
Sentiments: {0: 'negative', 1: 'neutral', 2: 'positive'}


## Weak-supervision aspect labeling

`assign_aspect` scores each review against per-aspect keyword lists
(`ASPECT_KEYWORDS`) and picks the strongest match — a labeling function, not
a model. This produces noisy-but-free aspect labels for reviews that only
come with a star rating.

In [3]:
examples = [
    "Great quality but shipping took 3 weeks",
    "Customer service was rude and unhelpful, total waste of money",
    "Easy to set up, intuitive interface and great instructions",
    "This thing is so durable, built like a tank",
]

for text in examples:
    aspect_id = assign_aspect(text)
    print(f"{text!r}\n  -> aspect: {config['aspects'][aspect_id]}")

'Great quality but shipping took 3 weeks'
  -> aspect: quality
'Customer service was rude and unhelpful, total waste of money'
  -> aspect: customer_service
'Easy to set up, intuitive interface and great instructions'
  -> aspect: usability
'This thing is so durable, built like a tank'
  -> aspect: quality


## Star rating -> sentiment bucketing

`stars_to_sentiment` maps the review's 1-5 star rating to
negative / neutral / positive.

In [4]:
for stars in range(1, 6):
    sentiment_id = stars_to_sentiment(stars)
    print(f"{stars} stars -> {config['sentiments'][sentiment_id]}")

1 stars -> negative
2 stars -> negative
3 stars -> neutral
4 stars -> positive
5 stars -> positive


## Loading a small live sample (optional, needs network)

`load_and_preprocess` streams from the configured Hugging Face dataset and
applies the labeling functions above. This overrides `train_size` to a tiny
number just for the demo — the real pipeline uses `config.yaml`'s
`data.train_size` (5000).

In [5]:
import copy

from src.data import load_and_preprocess

small_cfg = copy.deepcopy(config)
small_cfg["data"]["train_size"] = 20

try:
    texts, aspects, sentiments = load_and_preprocess(small_cfg, split="train")
    for text, aspect_id, sentiment_id in list(zip(texts, aspects, sentiments))[:5]:
        print(f"[{config['aspects'][aspect_id]} / {config['sentiments'][sentiment_id]}] {text[:100]}")
except Exception as e:
    print(f"Skipping live dataset download ({type(e).__name__}: {e})")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

C:\Users\ausku\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ausku\.cache\huggingface\hub\datasets--McAuley-Lab--Amazon-Reviews-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Amazon-Reviews-2023.py: 0.00B [00:00, ?B/s]

Streaming load failed (Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py), trying non-streaming...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Skipping live dataset download (RuntimeError: Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py)


## Short training run (CPU-feasible)

Builds a fresh model and runs `Trainer` for one epoch on a tiny slice of
data — enough to see the training loop (forward, joint loss, backward,
checkpointing) execute end to end, not to produce a usable model. A real
run is `python main.py train` with the full `config.yaml` sizes.

In [6]:
from src.data import create_dataloaders
from src.model import build_model
from src.train import Trainer

train_cfg = copy.deepcopy(config)
train_cfg["data"]["train_size"] = 20
train_cfg["data"]["val_size"] = 10
train_cfg["data"]["test_size"] = 10
train_cfg["training"]["epochs"] = 1
train_cfg["training"]["batch_size"] = 4
train_cfg["training"]["fp16"] = False
train_cfg["paths"]["model_dir"] = "models_demo/"
train_cfg["paths"]["results_dir"] = "results_demo/"

try:
    dataloaders = create_dataloaders(train_cfg)
    model = build_model(train_cfg)
    trainer = Trainer(model, train_cfg, dataloaders["train"], dataloaders["val"])
    history = trainer.train()
    print("Final train loss:", history["train_loss"][-1])
    print("Final val loss:  ", history["val_loss"][-1])
except Exception as e:
    print(f"Skipping live training demo ({type(e).__name__}: {e})")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Streaming load failed (Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py), trying non-streaming...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Skipping live training demo (RuntimeError: Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py)


## Next step

[03_v2_v3_feature_demos.ipynb](03_v2_v3_feature_demos.ipynb) — sarcasm
routing, quantum uncertainty, span extraction, and the V3 hybrid backbone.